# Anotación: dejar marcas sobre la imagen

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura oficial:** Procesamiento de Imágenes
* **Nombre de trabajo:** Laboratorio de Tecnologías de la Imagen Digital

---

## Objetivo

Marcar regiones y escribir etiquetas sobre una imagen con las funciones de dibujo de OpenCV, cuidando tres detalles que suelen producir errores: el orden de las coordenadas, el orden de los colores y los caracteres que las fuentes de OpenCV no saben dibujar.

## Al terminar este cuaderno van a poder:

1. Dibujar líneas, rectángulos y círculos indicando las coordenadas en orden `(x, y)`.
2. Dibujar sobre una copia para conservar la imagen original intacta.
3. Explicar por qué el mismo color se ve distinto según la imagen esté en RGB o en BGR.
4. Escribir texto con `cv2.putText` y medirlo con `cv2.getTextSize` para ubicarlo.
5. Reconocer que las fuentes de OpenCV no dibujan tildes ni eñes, y elegir una alternativa.
6. Anotar varias regiones descriptas como cajas `(x, y, ancho, alto)` y guardar el resultado.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Coordenada `(x, y)`** | La posición de un punto: `x` cuenta columnas hacia la derecha, `y` cuenta filas hacia abajo. |
| **Origen** | El punto `(0, 0)`: la esquina superior izquierda de la imagen. |
| **Grosor** | El ancho del trazo en píxeles. Un grosor negativo significa figura rellena. |
| **Antialiasing** | Suavizado de los bordes de un trazo para que no se vean escalonados. |
| **Caja delimitadora** | Un rectángulo que encierra un objeto, descripto por su esquina, su ancho y su alto. |
| **Etiqueta** | Texto que acompaña una marca para decir qué representa. |

## Marcar es comunicar un resultado

**Analogía.** Cuando alguien corrige un texto en papel, no reescribe todo: subraya, encierra en un círculo y anota al margen. Las funciones de dibujo cumplen ese papel sobre una imagen: señalan dónde está lo importante y qué es.

**Bajada tecnológica.** Todos los detectores que van a ver más adelante —de rostros, de objetos, de manos— devuelven sus resultados como cajas y puntos. Para mostrar esos resultados, o para revisar si un detector se equivocó, hay que dibujarlos sobre la imagen. También se usan para preparar ejemplos etiquetados a mano para entrenar modelos.

### ⚙ Configuración del entorno

In [ ]:
from pathlib import Path

import cv2
import numpy as np

from utils import mostrar_imágenes

print(f"OpenCV {cv2.__version__} — NumPy {np.__version__}")

Retomamos `leer_imagen_rgb` y preparamos la carpeta de salidas, donde vamos a guardar la imagen anotada al final.

In [ ]:
CARPETA_IMÁGENES = Path("..") / "imagenes"
CARPETA_SALIDAS = Path("salidas")
CARPETA_SALIDAS.mkdir(exist_ok=True)

def leer_imagen_rgb(ruta_de_la_imagen):
    """Lee una imagen con OpenCV, verifica la lectura y la devuelve en RGB."""
    imagen_leída = cv2.imread(filename=str(ruta_de_la_imagen), flags=cv2.IMREAD_COLOR)
    assert imagen_leída is not None, f"No se pudo leer: {ruta_de_la_imagen}"
    imagen_convertida = cv2.cvtColor(src=imagen_leída, code=cv2.COLOR_BGR2RGB)
    return imagen_convertida

print("Carpetas y función de lectura listas.")

## 1. Coordenadas: primero `x`, después `y`

**Analogía.** En un mapa o en la batalla naval se dice primero la columna y después la fila: "B-4". Las funciones de dibujo de OpenCV hablan así: `(x, y)`, columna primero.

**Bajada tecnológica.** Es la convención de los gráficos por computadora y de casi todos los formatos de anotación de imágenes. El problema es que el slicing de NumPy, que usamos para recortar, va al revés: `imagen[fila, columna]`. Confundir los dos órdenes es uno de los errores más comunes al dibujar resultados.

Practicamos sobre un lienzo negro, donde los trazos se ven con claridad. Un lienzo es simplemente un arreglo lleno de ceros.

### ✎ Antes de ejecutar

El lienzo va a tener 300 filas y 500 columnas. Vamos a dibujar un rectángulo desde el punto `(20, 80)` hasta el `(200, 260)`. ¿Va a quedar más cerca del borde izquierdo o del borde superior del lienzo?

In [ ]:
# 'zeros' crea un arreglo lleno de ceros: una imagen negra de 300 filas y 500 columnas.
lienzo = np.zeros(shape=(300, 500, 3), dtype=np.uint8)

# Cada función recibe la imagen, los puntos (x, y), un color (R, G, B) y un grosor.
cv2.line(img=lienzo, pt1=(20, 40), pt2=(480, 40), color=(255, 200, 0), thickness=4)
cv2.rectangle(img=lienzo, pt1=(20, 80), pt2=(200, 260), color=(0, 255, 0), thickness=3)
cv2.circle(img=lienzo, center=(350, 170), radius=80, color=(255, 0, 255), thickness=-1)

paneles = [{"imagen": lienzo, "título": "Línea, rectángulo y círculo relleno"}]
mostrar_imágenes(paneles, alto_de_figura=5)

La línea cruza el lienzo de izquierda a derecha a 40 filas del borde superior. El rectángulo empieza a 20 columnas del borde izquierdo y a 80 filas del borde superior. El círculo tiene grosor `-1`, así que se dibujó relleno.

Confirmamos el orden de las coordenadas leyendo el lienzo con indexación de NumPy. La esquina del rectángulo está en el punto `(x=20, y=80)`: para NumPy eso es la **fila 80, columna 20**.

In [ ]:
color_en_fila_80_columna_20 = lienzo[80, 20]
color_en_fila_20_columna_80 = lienzo[20, 80]

print(f"lienzo[80, 20] → {color_en_fila_80_columna_20}  (esquina del rectángulo)")
print(f"lienzo[20, 80] → {color_en_fila_20_columna_80}  (el mismo par de números, al revés)")

En `lienzo[80, 20]` aparece el verde del rectángulo; en `lienzo[20, 80]`, negro. Los mismos dos números, en distinto orden, señalan puntos distintos. Regla práctica: **para dibujar, `(x, y)`; para indexar, `[fila, columna]`**.

### ✎ Para pensar

- Si en el cuaderno `004_encuadre` recortaron `imagen[500:680, 180:380]`, ¿qué dos puntos le pasarían a `cv2.rectangle` para marcar exactamente esa región?
- ¿Por qué el grosor negativo tiene sentido como forma de pedir "relleno"?

## 2. Dibujar modifica la imagen

**Analogía.** Dibujar con OpenCV es escribir con birome sobre la foto impresa, no sobre una fotocopia: la marca queda en el original.

**Bajada tecnológica.** Un error frecuente en pipelines de análisis es dibujar los resultados sobre la misma imagen que después se usa para medir. Las marcas contaminan la medición siguiente, y el error es difícil de detectar.

Todas las funciones de dibujo trabajan **en el lugar**: modifican el arreglo que reciben. Para conservar el original, se dibuja sobre una copia. Lo comprobamos con la imagen de las cartas.

In [ ]:
cartas_originales = leer_imagen_rgb(CARPETA_IMÁGENES / "cards.jpg")

# 'copy' crea un arreglo nuevo con los mismos valores, en otra zona de memoria.
cartas_anotadas = cartas_originales.copy()
cv2.rectangle(img=cartas_anotadas, pt1=(110, 190), pt2=(352, 485), color=(255, 0, 0), thickness=4)

cambió_la_copia = not np.array_equal(a1=cartas_originales, a2=cartas_anotadas)
cartas_releídas = leer_imagen_rgb(CARPETA_IMÁGENES / "cards.jpg")
original_intacto = np.array_equal(a1=cartas_originales, a2=cartas_releídas)

print(f"¿La copia tiene el dibujo? {cambió_la_copia}")
print(f"¿El original sigue igual al archivo? {original_intacto}")

La copia cambió y el original quedó idéntico a lo que hay en el archivo. Si hubiéramos dibujado directamente sobre `cartas_originales`, el rectángulo habría quedado en esa variable para el resto del cuaderno.

## 3. El color depende del orden de los canales

**Analogía.** Si en un semáforo alguien intercambia de lugar la luz roja y la verde, las lámparas funcionan perfecto, pero el mensaje se invierte. Una tupla de color en OpenCV es solo una lista de tres números: qué color representa depende del orden de canales de la imagen.

**Bajada tecnológica.** Es un error clásico en proyectos reales: el sistema dibuja las cajas de detección en rojo, pero en el archivo guardado aparecen azules. No falla el dibujo, sino la mezcla entre RGB y BGR.

### ✎ Antes de ejecutar

Sobre `cartas_anotadas`, que está en RGB, dibujamos un rectángulo con `color=(255, 0, 0)`. Ahora vamos a guardar esa imagen **sin convertirla** con `cv2.imwrite` y a volver a abrir el archivo. ¿De qué color esperan ver el rectángulo?

In [ ]:
ruta_sin_convertir = CARPETA_SALIDAS / "cartas_sin_convertir.png"

# 'imwrite' interpreta el arreglo como BGR, pero 'cartas_anotadas' está en RGB.
cv2.imwrite(filename=str(ruta_sin_convertir), img=cartas_anotadas)
cartas_desde_archivo = leer_imagen_rgb(ruta_sin_convertir)

paneles = [
    {"imagen": cartas_anotadas, "título": "En memoria (RGB): color=(255, 0, 0)"},
    {"imagen": cartas_desde_archivo, "título": "Guardada sin convertir y reabierta"},
]
mostrar_imágenes(paneles, alto_de_figura=5)

En memoria el rectángulo es rojo. En el archivo guardado sin convertir, el rectángulo aparece azul y, además, toda la foto tiene el rojo y el azul intercambiados: la madera se ve azulada y los palos rojos, azules. El número `255` quedó en la posición 0, que para OpenCV es el azul.

**Regla práctica:** la tupla de color sigue el orden de canales de la imagen sobre la que se dibuja, y antes de guardar con `imwrite` hay que volver a BGR con `cv2.cvtColor(src=imagen, code=cv2.COLOR_RGB2BGR)`.

## 4. Escribir texto

**Analogía.** Un rótulo en un mapa tiene que estar cerca de lo que nombra, sin taparlo, y con un fondo que lo haga legible sobre cualquier color.

**Bajada tecnológica.** Las cámaras de seguridad sobreimprimen la fecha y la hora; los sistemas de detección escriben el nombre del objeto y su nivel de confianza junto a cada caja.

`cv2.putText` tiene varios parámetros que cambian el resultado:

| Parámetro | Qué significa | Qué cambia si lo modificamos |
|---|---|---|
| `org` | Punto `(x, y)` de la esquina **inferior** izquierda del texto | Mueve el texto; si `y` es muy chico, el texto queda cortado arriba |
| `fontFace` | Tipografía, por ejemplo `cv2.FONT_HERSHEY_SIMPLEX` | Cambia el estilo de las letras |
| `fontScale` | Factor de tamaño de la letra | Agranda o achica el texto |
| `thickness` | Grosor del trazo | Letras más finas o más gruesas |
| `lineType` | `cv2.LINE_AA` suaviza los bordes | Sin él, las letras se ven escalonadas |

Para ubicar una etiqueta con fondo necesitamos saber cuánto mide el texto antes de dibujarlo. Eso lo calcula `cv2.getTextSize`.

In [ ]:
texto_de_la_etiqueta = "Rey de picas"
tipografía = cv2.FONT_HERSHEY_SIMPLEX
escala_de_letra = 0.8
grosor_de_letra = 2

# 'getTextSize' devuelve ((ancho, alto), línea_base) del texto, sin dibujarlo.
medidas_del_texto = cv2.getTextSize(text=texto_de_la_etiqueta, fontFace=tipografía, fontScale=escala_de_letra, thickness=grosor_de_letra)
tamaño_del_texto = medidas_del_texto[0]
ancho_del_texto = tamaño_del_texto[0]
alto_del_texto = tamaño_del_texto[1]

print(f"El texto '{texto_de_la_etiqueta}' ocupa {ancho_del_texto} × {alto_del_texto} píxeles")

Con esas medidas podemos dibujar primero un rectángulo relleno del tamaño justo y, encima, el texto. Lo ubicamos pegado al borde superior del recuadro de la carta.

In [ ]:
cartas_con_etiqueta = cartas_anotadas.copy()
x_de_la_etiqueta = 110
y_de_la_etiqueta = 190

fondo_superior_izquierdo = (x_de_la_etiqueta, y_de_la_etiqueta - alto_del_texto - 12)
fondo_inferior_derecho = (x_de_la_etiqueta + ancho_del_texto + 10, y_de_la_etiqueta)
cv2.rectangle(img=cartas_con_etiqueta, pt1=fondo_superior_izquierdo, pt2=fondo_inferior_derecho, color=(255, 0, 0), thickness=-1)

posición_del_texto = (x_de_la_etiqueta + 5, y_de_la_etiqueta - 6)
cv2.putText(img=cartas_con_etiqueta, text=texto_de_la_etiqueta, org=posición_del_texto, fontFace=tipografía, fontScale=escala_de_letra, color=(255, 255, 255), thickness=grosor_de_letra, lineType=cv2.LINE_AA)

paneles = [{"imagen": cartas_con_etiqueta, "título": "Recuadro con etiqueta"}]
mostrar_imágenes(paneles, alto_de_figura=6)

La etiqueta queda apoyada sobre el borde superior del recuadro, con letras blancas sobre fondo rojo. Como calculamos el ancho del texto, el fondo tiene exactamente el tamaño necesario, sin cortar letras ni sobrar espacio.

### ✎ Modificación guiada

En la celda anterior, cambien `escala_de_letra` por **1.2** en la celda de `getTextSize` y vuelvan a ejecutar las dos celdas. Después cambien `x_de_la_etiqueta` e `y_de_la_etiqueta` para mover la etiqueta a la esquina inferior del recuadro. Antes de ejecutar, anoten qué valor de `y` necesitan.

### Una limitación importante: tildes y eñes

Las tipografías que trae OpenCV solo incluyen caracteres básicos, sin tildes ni eñes. Lo comprobamos escribiendo dos palabras en español sobre un lienzo.

In [ ]:
lienzo_de_texto = np.zeros(shape=(120, 560, 3), dtype=np.uint8)

cv2.putText(img=lienzo_de_texto, text="Corazones", org=(15, 50), fontFace=tipografía, fontScale=1.2, color=(255, 255, 255), thickness=2, lineType=cv2.LINE_AA)
cv2.putText(img=lienzo_de_texto, text="Año, diamante, única", org=(15, 100), fontFace=tipografía, fontScale=1.2, color=(255, 255, 255), thickness=2, lineType=cv2.LINE_AA)

paneles = [{"imagen": lienzo_de_texto, "título": "putText con y sin caracteres especiales"}]
mostrar_imágenes(paneles, alto_de_figura=3)

La primera línea se ve bien. En la segunda, cada letra con tilde o eñe se reemplazó por signos de pregunta: OpenCV no sabe dibujarlas. Hay tres salidas razonables:

| Alternativa | Ventaja | Costo |
|---|---|---|
| Escribir sin tildes en la imagen ("Ano", "unica") | Simple, no requiere nada más | El texto queda con errores ortográficos visibles |
| Usar etiquetas cortas o códigos ("P1", "K-picas") | Evita el problema y ocupa poco lugar | Hace falta una leyenda aparte |
| Poner los textos en el título o la leyenda del gráfico de Matplotlib | Matplotlib sí dibuja tildes | El texto no queda grabado en el archivo de imagen |

En este curso preferimos la segunda o la tercera opción, para no publicar texto con errores.

### ✎ Para pensar

- ¿Por qué el punto `org` de `putText` indica la esquina inferior del texto y no la superior, como el rectángulo? Piensen en cómo se escribe sobre un renglón.
- Si una etiqueta se dibuja cerca del borde superior de la imagen, ¿qué puede pasar con el fondo calculado? ¿Cómo lo evitarían?

## 5. Anotar varias regiones

**Analogía.** Un plano de evacuación marca varias zonas con el mismo estilo: recuadro, color y rótulo. Si cada zona se dibujara de una manera distinta, el plano sería difícil de leer.

**Bajada tecnológica.** Los detectores devuelven listas de cajas con el formato `(x, y, ancho, alto)`. Es el mismo formato que devuelve `cv2.boundingRect`, que van a usar en el nivel fundamental con contornos, y el que usan los detectores de rostros y de objetos del nivel avanzado.

Como ya entendemos el paso a paso —medida del texto, fondo y texto—, lo encapsulamos en dos funciones. La primera se ocupa solo de la etiqueta.

In [ ]:
def dibujar_etiqueta(imagen, texto, x, y, color):
    """Escribe un texto blanco sobre un fondo de color, apoyado sobre el punto (x, y)."""
    medidas = cv2.getTextSize(text=texto, fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1.0, thickness=2)
    tamaño = medidas[0]
    ancho_del_texto = tamaño[0]
    alto_del_texto = tamaño[1]

    fondo_superior = (x, y - alto_del_texto - 14)
    fondo_inferior = (x + ancho_del_texto + 12, y)
    cv2.rectangle(img=imagen, pt1=fondo_superior, pt2=fondo_inferior, color=color, thickness=-1)

    posición_del_texto = (x + 6, y - 8)
    cv2.putText(img=imagen, text=texto, org=posición_del_texto, fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1.0, color=(255, 255, 255), thickness=2, lineType=cv2.LINE_AA)

La segunda recibe una región descripta como diccionario, dibuja su recuadro y le pone la etiqueta usando la función anterior.

In [ ]:
def anotar_región(imagen, región, color):
    """Dibuja el recuadro de una región y su etiqueta sobre la misma imagen.

    'región' es un diccionario con 'nombre', 'x', 'y', 'ancho' y 'alto' en píxeles.
    """
    esquina_superior = (región["x"], región["y"])
    esquina_inferior = (región["x"] + región["ancho"], región["y"] + región["alto"])
    cv2.rectangle(img=imagen, pt1=esquina_superior, pt2=esquina_inferior, color=color, thickness=4)
    dibujar_etiqueta(imagen=imagen, texto=región["nombre"], x=región["x"], y=región["y"], color=color)

Ninguna de las dos funciones devuelve nada: dibujan sobre la imagen que reciben, igual que las funciones de OpenCV. Por eso las vamos a llamar siempre sobre una copia.

Describimos tres regiones de la foto de los botes como una lista de diccionarios. La región de los nenúfares usa las mismas coordenadas del recorte de `004_encuadre`, traducidas al formato de caja.

In [ ]:
regiones_de_los_botes = [
    {"nombre": "P1", "x": 400, "y": 200, "ancho": 150, "alto": 240},
    {"nombre": "P2", "x": 700, "y": 370, "ancho": 180, "alto": 200},
    {"nombre": "Nenufares", "x": 180, "y": 500, "ancho": 200, "alto": 180},
]

# El recorte [500:680, 180:380] empieza en la columna 180 (x) y en la fila 500 (y).
cantidad_de_regiones = len(regiones_de_los_botes)
print(f"Regiones a anotar: {cantidad_de_regiones}")

El recorte `[500:680, 180:380]` se traduce a una caja con `x=180`, `y=500`, `ancho=200` y `alto=180`. Usamos "P1" y "P2" para las dos personas, y escribimos "Nenufares" sin tilde porque, como vimos, OpenCV no puede dibujarla. La aclaración con tildes va en el título de la figura.

In [ ]:
botes_originales = leer_imagen_rgb(CARPETA_IMÁGENES / "small-boats_1.jpg")
botes_anotados = botes_originales.copy()

for región in regiones_de_los_botes:
    anotar_región(imagen=botes_anotados, región=región, color=(230, 60, 0))

paneles = [{"imagen": botes_anotados, "título": "P1 y P2: personas en el bote — Nenufares: nenúfares"}]
mostrar_imágenes(paneles, alto_de_figura=7)

Las tres regiones quedan marcadas con el mismo estilo. Las cajas P1 y P2 encierran a cada una de las personas del bote, y la tercera coincide con el grupo de nenúfares que recortamos en el cuaderno anterior. La leyenda con tildes quedó en el título de la figura, que dibuja Matplotlib.

Guardamos el producto del cuaderno. Como la imagen está en RGB, la convertimos a BGR antes de `imwrite`.

In [ ]:
botes_anotados_bgr = cv2.cvtColor(src=botes_anotados, code=cv2.COLOR_RGB2BGR)
ruta_anotada = CARPETA_SALIDAS / "botes_anotados.png"

se_guardó = cv2.imwrite(filename=str(ruta_anotada), img=botes_anotados_bgr)
assert se_guardó, "No se pudo guardar la imagen anotada."

print(f"Imagen anotada guardada en: {ruta_anotada}")

El archivo `salidas/botes_anotados.png` conserva las marcas con los colores correctos, porque esta vez convertimos a BGR antes de guardar.

### ✎ Completado

Agreguen una cuarta región para el motor fuera de borda, que está en el extremo derecho del bote, y vuelvan a anotar la imagen.

In [ ]:
# ─── Completen los pasos faltantes ───────────────────────────────────────────
#
# TODO 1: miren la figura y estimen x, y, ancho y alto del motor.
# TODO 2: agreguen un diccionario con nombre "Motor" a 'regiones_de_los_botes'.
# TODO 3: vuelvan a ejecutar la celda que anota y muestra la imagen.
#
# regiones_de_los_botes.append({"nombre": "Motor", "x": ..., "y": ..., "ancho": ..., "alto": ...})
#
# ─────────────────────────────────────────────────────────────────────────────

## Actividad de producción

In [ ]:
# ─── Espacio de práctica ─────────────────────────────────────────────────────
#
# Consigna: anoten las cuatro cartas de cards.jpg.
#   1. Carguen la imagen y creen una copia para dibujar.
#   2. Describan cada carta como una región con nombre, x, y, ancho y alto.
#   3. Anótenlas con anotar_región, usando un color distinto por palo.
#   4. Guarden el resultado en 'salidas' (recuerden convertir a BGR).
#   5. Escriban en una celda de texto cómo decidieron dónde ubicar cada
#      etiqueta para que no tape la carta vecina.
#
# ─────────────────────────────────────────────────────────────────────────────

## Cierre

Anotar una imagen es sencillo, pero tiene tres trampas que conviene recordar siempre.

| Tema | Qué recordar |
|---|---|
| Coordenadas | Para dibujar se usa `(x, y)`; para indexar, `[fila, columna]`. |
| Modificación | Las funciones de dibujo cambian el arreglo que reciben: dibujar sobre una copia. |
| Color | La tupla sigue el orden de canales de la imagen; convertir a BGR antes de guardar. |
| Texto | `org` es la esquina inferior izquierda; `getTextSize` permite calcular el fondo. |
| Caracteres | Las fuentes de OpenCV no dibujan tildes ni eñes. |

**Limitaciones de lo que vimos.** Las coordenadas de todas las regiones las estimamos mirando la imagen: son aproximadas y no se adaptan si la foto cambia. En el nivel fundamental vamos a obtener cajas automáticamente a partir de contornos, y en el avanzado, a partir de detectores. Además, la tipografía de OpenCV es muy limitada; para textos elaborados conviene otra herramienta.

En el próximo cuaderno, `007_ficha`, integramos todo lo del nivel: vamos a tomar una imagen propia, diagnosticarla, encuadrarla, anotarla y convertir el proceso en un script.